<a href="https://colab.research.google.com/github/helenamrch/OAS5900-Masteroppgave/blob/main/Extract_participants.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Participant Extractor – Ungdomsråd protokoller

Reads **all `.md` files** in every subfolder, extracts meeting participants,
and outputs a CSV with columns: `municipality`, `date`, `name`, `status`.

### Two-pass approach for concatenated name lines
Some files (PDF-to-markdown artefact) write multiple names on a single line:
> 'PLACEHOLDER'

The notebook handles this with a **growing `known_names` set**:
1. **Pass 1** – scan every file; collect all names that appear alone on a line
   into `known_names` (no CSV output yet).
2. **Pass 2** – re-parse every file; for long ambiguous lines use
   greedy longest-match against `known_names` to split them.
   `known_names` keeps growing as pass 2 progresses.

### Supported file formats
| Format | Examples |
|--------|----------|
| Standard Navn/Funksjon table | 'PLACEHOLDER' |
| Attendance grid (alternating name/marker columns) | 'PLACEHOLDER' |
| Multi-column label+name grid with inline section changes | 'PLACEHOLDER' |
| Bullet/plain list under section headers | 'PLACEHOLDER' |
| Concatenated names on one line | 'PLACEHOLDER' |
| `Lastname, Firstname` format (optionally concatenated) | 'PLACEHOLDER' |
| Nynorsk headings (`Møtande medlemmar`, `Følgjande medlem møtte`) | 'PLACEHOLDER' |
| Split headings across two lines | 'PLACEHOLDER' |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── Cell 1 ─ Imports & configuration ────────────────────────────────────────
import re
from pathlib import Path
import pandas as pd

BASE_DIR   = Path(r"'PLACEHOLDER'")
OUTPUT_CSV = BASE_DIR / "'PLACEHOLDER'"

# ── Manual exclusion list ────────────────────────────────────────────────────
# Add any string that is being wrongly identified as a participant name.
# Matching is exact and case-sensitive.
# Uncomment an example line or add new entries between the braces.
EXCLUDED_NAMES: set = {
    # "Ungdomsrådet sitt forslag vart samrøystes vedtatt",
    # "Møreaksen er viktig",
}

print("Base dir:", BASE_DIR)
print("Output  :", OUTPUT_CSV)
print(f"Excluded names: {len(EXCLUDED_NAMES)} entries")

In [ ]:
# ── Cell 2 ─ Discover subfolders and ALL .md files in each ───────────────────

md_files = {}   # {municipality_name: [Path, ...]}
for sub in sorted(BASE_DIR.iterdir()):
    if sub.is_dir():
        files = sorted(sub.glob("*.md"))
        if files:
            md_files[sub.name] = files

total_files = sum(len(v) for v in md_files.values())
print(f"Found {len(md_files)} folders  |  {total_files} .md files total\n")
for name, paths in md_files.items():
    print(f"  {name:<35}  {len(paths):2d} file(s)")

In [ ]:
# ── Cell 3 ─ Date extraction + OCR normalisation ─────────────────────────────

_MONTHS = {
    'januar':'01', 'februar':'02', 'mars':'03',    'april':'04',
    'mai':'05',    'juni':'06',    'juli':'07',    'august':'08',
    'september':'09','oktober':'10','november':'11','desember':'12',
}
_DATE_NUM  = re.compile(r'(\d{1,2})\.(\d{1,2})\.(\d{4})')
_DATE_WORD = re.compile(
    r'(\d{1,2})\.?\s*(' + '|'.join(_MONTHS) + r')(?:\s+(\d{4}))?',
    re.IGNORECASE,
)

def extract_date(text: str) -> str:
    """Return ISO date YYYY-MM-DD found in *text*, or empty string."""
    m = _DATE_NUM.search(text)
    if m:
        return f"{m.group(3)}-{m.group(2).zfill(2)}-{m.group(1).zfill(2)}"
    m = _DATE_WORD.search(text)
    if m:
        yr = m.group(3) or "????"
        return f"{yr}-{_MONTHS[m.group(2).lower()]}-{m.group(1).zfill(2)}"
    return ""


def normalise_ocr(text: str) -> str:
    """
    Fix common PDF-to-markdown OCR encoding artefacts before parsing.

    1. Polish stroke letters ł/Ł → Norwegian ø/Ø
       (frequent OCR confusion — affects 'PLACEHOLDER' headings).
    2. Single uppercase letter split from its continuation by spaces
       e.g. 'M  edlem  mer' → 'Medl emmer' (then collapsed by step 3)
            'V  aramedlem' → 'Varamedlem'
            'S kjold'      → 'Skjold'
    3. Collapse runs of two or more spaces → single space.
    """
    # 1. ł (U+0142) → ø,  Ł (U+0141) → Ø
    text = text.replace('\u0142', '\xf8').replace('\u0141', '\xd8')
    # 2. Join lone uppercase letter to its lowercase continuation
    text = re.sub(r'\b([A-ZÆØÅ]) +([a-zæøå])', lambda m: m.group(1) + m.group(2), text)
    # 3. Collapse multiple spaces
    text = re.sub(r'  +', ' ', text)
    return text

In [ ]:
# ── Cell 4 ─ Name utilities ──────────────────────────────────────────────────

_NON_NAME_STARTS = {
    'Dato','Tid','Sted','Stad','Møtested','Utvalg','Arkivsak',
    'Parti','Funksjon','Navn','Stilling','Side','Kl','Ref',
    'Tlf','Epost','Tittel','Rolle','Representerer','Teams',
    'Tidspunkt','Møtetid','Møtedato','Merknad','Merknadar',
    'SAKLISTE','Sakliste','Sak','SAK','PS','RS','Pkt','Fra',
    'Sekretariat','Rådgiver','Koordinator','Sekretær',
    'Protokollfører','Møteleder','Ungdomskonsulent',
    'Varamedlem','Varamedlemmer','Varamedlemmar','Medlem',
    'Leder','Nestleder','Leiar','Nestleiar',
}

def clean_name(s: str) -> str:
    """Strip bullets, parenthetical annotations, trailing abbreviations."""
    s = re.sub(r'^[-–•*]\s*', '', s.strip())
    s = re.sub(r'\([^)]*\)', '', s)          # (grade), (vara), (2ANC), …
    s = re.sub(r'\s{2,}', ' ', s)
    # trailing ALL-CAPS school/location codes: "UNGR", "B", "SP", etc.
    # Fix Q: only strip ALL-CAPS sequences, not mixed-case names like "Olsen"
    s = re.sub(r'\s+[A-ZÆØÅ]{1,6}\s*$', '', s)
    return s.strip()

def is_name(s: str) -> bool:
    """
    Heuristic: does *s* look like a single person's name?
    Two to seven words, first word capitalised, no digits or sentence punctuation.
    """
    s = s.strip()
    if not s or len(s) < 4:
        return False
    words = s.split()
    if len(words) < 2 or len(words) > 7:
        return False
    if words[0] in _NON_NAME_STARTS:
        return False
    if not words[0][0].isupper():
        return False
    if re.search(r'\d', s):
        return False
    if re.search(r'[.,:;!?@#$%^&*+=\[\]{}<>/\\|]', s):
        return False
    return True

In [ ]:
# ── Cell 5 ─ Section detection ───────────────────────────────────────────────
#
# Returns one of: 'member' | 'vara' | 'absent' | 'admin' | 'end' | 'unknown'
#
# Nynorsk / OCR variants handled:
#   Møtande medlemmar         ← already covered
#   Følgjande/Følgande/Følgende med… møtte
#   Frammøtte / Frammøte      ← 'PLACEHOLDER' (framm, not fremm)
#   Desse representantane møtte  ← 'PLACEHOLDER', 'PLACEHOLDER'
#   Desse møtte               ← 'PLACEHOLDER' (Fix O)
#   Oppmøtte                  ← 'PLACEHOLDER' (Fix O)
#   Medlemmer / Medlemmar     ← standalone heading (Sola after OCR fix)
#   Varamedlem (standalone)   ← 'PLACEHOLDER' multi-column layout
#   ł→ø OCR fix covered by normalise_ocr() in Cell 3

_SEC_MEMBER = re.compile(
    r'm\xf8tande\s+medlemmar'                   # Nynorsk: Møtande medlemmar
    r'|m\xf8tende\s+med'                        # Bokmål:  Møtende med…
    r'|f\xf8lg(?:j?and|end)e\s+med.*m\xf8tte'  # Følgjande/Følgende/Følgande med… møtte
    r'|faste\s+med'
    r'|fremm\xf8tte|fremm\xf8te'               # Bokmål fremmøtte
    r'|framm\xf8tte|framm\xf8te'               # Nynorsk frammøtte  (Luster)
    r'|desse\s+representantane\s+m\xf8tte'     # 'PLACEHOLDER' / 'PLACEHOLDER'
    r'|\bmedlemmer\b|\bmedlemmar\b'             # standalone heading ('PLACEHOLDER')
    r'|til\s+stede'
    r'|tilstede'
    r'|deltakere'
    r'|representanter\s+som'
    r'|m\xf8te[dn]e\s+repr'
    r'|oppm\xf8tte'                            # Fix O: 'PLACEHOLDER' "Oppmøtte:"
    r'|desse\s+m\xf8tte',                      # Fix O: 'PLACEHOLDER' "Desse møtte:"
    re.I,
)
_SEC_VARA = re.compile(
    r'm\xf8tande\s+varamedlemmar'
    r'|f\xf8lg(?:j?and|end)e\s+vara.*m\xf8tte'
    r'|m\xf8tende\s+vara'
    r'|varamedlemm'                             # varamedlemmer / varamedlemmar
    r'|\bvaramedlem\b(?!\s+for)',               # standalone "Varamedlem" heading ('PLACEHOLDER')
    re.I,                                       # Fix L: (?!\s+for) avoids column header
)
_SEC_ABSENT = re.compile(
    r'\bforfall\b|m\xf8tte\s+ikke|meldt\s+forfall|ikke\s+m\xf8tt|frav\xe6r',
    re.I,
)
_SEC_ADMIN = re.compile(
    r'administrasjon|sekret\xe6r|r\xe5dgiver|koordinator'
    r'|ungdomskonsulent|m\xf8teleder|protokollf\xf8rer|sekretariat',
    re.I,
)
_SEC_END = re.compile(
    r'^#+\s*(sak|ps|rs)|^sakliste|orienteringssaker'
    r'|godkjenning\s+av\s+inn|godkjenning\s+av\s+sak'
    r'|^behandling$',                           # Fix K: removed ^arkivsak (metadata label)
    re.I,
)
_SPLIT_HDR_START = re.compile(
    r'^(?:m\xf8tand|m\xf8tend|varamedl(?!em\s+for)|fremm\xf8tt|framm\xf8tt'
    r'|til\s+stede|forfall|administrasjon'
    r'|f\xf8lgj?and|f\xf8lgend|desse|oppm\xf8tt)',   # Fix P: varamedlem for excluded; oppmøtt added
    re.I,
)

def classify(line: str) -> str:
    ll = line.lower().strip()
    if _SEC_END.search(ll):    return 'end'
    if _SEC_ADMIN.search(ll):  return 'admin'
    if _SEC_ABSENT.search(ll): return 'absent'
    if _SEC_VARA.search(ll):   return 'vara'
    if _SEC_MEMBER.search(ll): return 'member'
    # Second attempt with all spaces collapsed:
    # handles OCR-fragmented headings like 'M edlem mer' → 'Medlemmer'
    ll_c = re.sub(r'\s+', '', ll)
    if _SEC_MEMBER.search(ll_c): return 'member'
    if _SEC_VARA.search(ll_c):   return 'vara'
    return 'unknown'


In [ ]:
# ── Cell 6 ─ Name splitters ──────────────────────────────────────────────────

# Matches parenthesised ALL-CAPS group used as a name delimiter
# e.g. ''PLACEHOLDER' (UFR) 'PLACEHOLDER' (UFR) …'
_PAREN_DELIM = re.compile(r'\s*\([A-ZÆØÅ]{2,6}\)\s*')


def split_with_known_names(line: str, known: set) -> list:
    """
    Greedy longest-match split of *line* using *known* name set.
    Returns list of matched name strings;
    if no split is possible returns [line] unchanged.
    """
    line = line.strip()
    if not line:
        return []
    if line in known:
        return [line]

    words  = line.split()
    ranked = sorted(known, key=lambda x: len(x.split()), reverse=True)

    result = []
    i = 0
    while i < len(words):
        matched = False
        for name in ranked:
            nw = name.split()
            n  = len(nw)
            if words[i : i + n] == nw:
                result.append(name)
                i += n
                matched = True
                break
        if not matched:
            result.append(' '.join(words[i:]))   # unmatched remainder
            break

    return result if result else [line]


def split_last_first_line(line: str) -> list:
    """
    Parse a concatenated sequence of 'Lastname, Firstname' entries
    (Nesodden style) and return a list of 'Firstname Lastname' strings.

    Example::
        ''PLACEHOLDER', 'PLACEHOLDER', 'PLACEHOLDER'' -> [''PLACEHOLDER'', ''PLACEHOLDER'']
    """
    pattern = re.compile(
        r'([A-Z\xc6\xd8\xc5][\w\-]+(?:\s+[A-Z\xc6\xd8\xc5][\w\-]+)*?)'
        r',\s+'
        r'([A-Z\xc6\xd8\xc5][\w\-]+(?:\s+[A-Z\xc6\xd8\xc5][\w\-]+)*?)'
        r'(?=\s+[A-Z\xc6\xd8\xc5][\w\-]+(?:\s+[A-Z\xc6\xd8\xc5][\w\-]+)*,|\s*$)'
    )
    results = []
    for last, first in pattern.findall(line):
        name = f"{first.strip()} {last.strip()}"
        if is_name(name):
            results.append(name)
    return results

In [ ]:
# ── Cell 7 ─ Standard Navn/Funksjon table parser  ('PLACEHOLDER' / 'PLACEHOLDER' / 'PLACEHOLDER') ─
#
# Handles tables whose header row contains 'Navn', 'Name', or 'Medl…' as the
# name column, and optionally 'Funksjon' / 'Rolle' / 'Status' as the status col.

def parse_standard_table(lines: list) -> list:
    """
    Parse a markdown table with a name column + optional status column.
    Returns [(name, status), ...].
    """
    results   = []
    name_idx  = None
    func_idx  = None
    header_ok = False

    for line in lines:
        if '|' not in line:
            continue
        if re.match(r'^[\s|:\-]+$', line):
            continue

        cols = [c.strip() for c in line.split('|')]
        while cols and not cols[0]:  cols.pop(0)
        while cols and not cols[-1]: cols.pop()

        if not header_ok:
            for j, col in enumerate(cols):
                cl = col.lower()
                # 'Navn', 'Name', or 'Medl…' ('PLACEHOLDER': 'Medlemmer:', 'PLACEHOLDER': 'Medlemmer:')
                # Fix R: skip 'medl' match if column also says 'vara' (e.g. "Varamedlem for")
                if 'navn' in cl or 'name' in cl or ('medl' in cl and 'vara' not in cl):
                    name_idx = j
                if 'funksjon' in cl or 'rolle' in cl or 'status' in cl:
                    func_idx = j
            header_ok = True
            continue

        ni   = name_idx if name_idx is not None else 0
        raw  = cols[ni] if ni < len(cols) else ''
        name = clean_name(raw)
        if not is_name(name):
            continue

        if func_idx is not None and func_idx < len(cols):
            status = 'varamedlem' if 'vara' in cols[func_idx].lower() else 'medlem'
        else:
            status = 'ukjent'

        results.append((name, status))

    return results

In [ ]:
# ── Cell 7b ─ Label-row table parser  ('PLACEHOLDER' style) ──────────────────
#
# Format: a 2-column table where col[0] is a section label and col[1] holds
# a comma-separated name list, optionally with a 'Varamedlemmer:' sub-label.
#
# Example:
#   | Til stede | Medlemmer: A, B, C  Varamedlemmer: D |
#   | Forfall   | E, F, G                               |

def parse_label_row_table(lines: list) -> list:
    """
    Returns [(name, status), ...].
    Skips 'Forfall' / 'Administrasjonen' rows.
    """
    results = []

    for line in lines:
        if '|' not in line:
            continue
        if re.match(r'^[\s|:\-]+$', line):
            continue

        cols = [c.strip() for c in line.split('|')]
        while cols and not cols[0]:  cols.pop(0)
        while cols and not cols[-1]: cols.pop()
        if len(cols) < 2:
            continue

        row_sec = classify(cols[0])
        if row_sec not in ('member', 'vara'):
            continue

        names_cell = cols[1]

        # Split on 'Varamedlemmer:' sub-label if present
        vara_split = re.split(r'varamedlemm\w*\s*:', names_cell, flags=re.I)
        member_text = vara_split[0]
        vara_text   = vara_split[1] if len(vara_split) > 1 else ''

        # Strip leading 'Medlemmer:' label from the member portion
        member_text = re.sub(r'^medl\w*\s*:', '', member_text, flags=re.I).strip()

        for raw in member_text.split(','):
            name = clean_name(raw)
            if is_name(name):
                results.append((name, 'varamedlem' if row_sec == 'vara' else 'medlem'))

        for raw in vara_text.split(','):
            name = clean_name(raw)
            if is_name(name):
                results.append((name, 'varamedlem'))

    return results

In [ ]:
# ── Cell 8 ─ Attendance-grid table parser  ('PLACEHOLDER' / 'PLACEHOLDER' style) ────────

def parse_attendance_grid(lines: list) -> list:
    """
    Handle two multi-column attendance table variants.

    **'PLACEHOLDER' style** – repeating (Name, marker) column pairs:
        | Name (grade) School | x | Name | x | ... |
    A row is present only when the marker column contains a non-empty value.

    **'PLACEHOLDER' style** – label column + name columns with
    inline section changes ("Varamedlemmer:", "Meldt forfall:"):
        | Medlemmer: | Varamedlemmer: | Selma … | Selma … | school |
    """
    results      = []
    seen         = set()          # deduplicate ('PLACEHOLDER' repeats names)
    section      = 'member'
    header_done  = False
    is_alternating = None         # True = 'PLACEHOLDER', False = 'PLACEHOLDER'

    for line in lines:
        if '|' not in line:
            continue
        if re.match(r'^[\s|:\-]+$', line):
            continue

        cols = [c.strip() for c in line.split('|')]
        while cols and not cols[0]:  cols.pop(0)
        while cols and not cols[-1]: cols.pop()

        if not header_done:
            header_done = True
            til_count = sum(
                1 for c in cols
                if re.search(r'til\s*stede|tilstede', c, re.I)
            )
            is_alternating = (til_count >= 2)
            continue

        # ── 'PLACEHOLDER' style ──────────────────────────────────────────────
        if not is_alternating:
            for cell in cols:
                sec = classify(cell)
                if sec in ('member', 'vara', 'absent'):
                    section = sec

            if section == 'absent':
                continue

            for cell in cols:
                name = clean_name(cell)
                if classify(name) != 'unknown':   # skip cells that are section labels
                    continue
                if is_name(name) and name not in seen:
                    seen.add(name)
                    status = 'varamedlem' if section == 'vara' else 'medlem'
                    results.append((name, status))

        # ── 'PLACEHOLDER' alternating style ──────────────────────────────────────
        else:
            i = 0
            while i + 1 < len(cols):
                raw_name   = cols[i]
                raw_marker = cols[i + 1]
                i += 2
                if not raw_marker.strip():   # empty marker = absent
                    continue
                name = clean_name(raw_name)
                if is_name(name) and name not in seen:
                    seen.add(name)
                    results.append((name, 'medlem'))

    return results

In [ ]:
# ── Cell 9 ─ Core file parser ────────────────────────────────────────────────
#
# Fixes applied vs. previous version
# ────────────────────────────────────────────────────────────────────────────
# A  has_navn_table also detects 'Medl…' header ('PLACEHOLDER', 'PLACEHOLDER')
# B  has_label_row_table detects 'Til stede | names' tables ('PLACEHOLDER')
# C  normalise_ocr() applied to raw text ('PLACEHOLDER')
# D  Headerless fallback: comma-name line before any section header ('PLACEHOLDER')
# E  Comma-split: simple 'First Last, First Last' tried first, then Last,First
# F  names_seen guard: don't leave 'member' before first name is parsed
# G  (ABBREV) group delimiter split ('PLACEHOLDER': 'Name (UFR) Name (UFR) …')
# H  Lookahead: peek at next non-empty line for inline 'Varamedlem' status
#    ('PLACEHOLDER': name on one line, role on next)
# I  Space-concat without delimiters ('PLACEHOLDER'): best-effort
# J  pending_hdr fall-through: don't discard names line after header completion
#    ('PLACEHOLDER': "Til stede:" header followed by comma-separated names)
# K  Removed ^arkivsak from _SEC_END (metadata label, not agenda) — cell-5
# L  (?!\s+for) on \bvaramedlem\b in _SEC_VARA (column header) — cell-5
# M  Lookahead also detects 'Medlem'/'Leder'/'Nestleder' roles on next line
#    ('PLACEHOLDER'/'PLACEHOLDER': role label on separate line after each name)
# N  has_navn_table: check individual cell length < 25 to avoid matching
#    agenda body text containing "medlemmene" ('PLACEHOLDER')

def _emit(records, collect_pass, municipality, date, name, status, known_names):
    """Helper: add name to known_names and optionally append a record."""
    known_names.add(name)
    if not collect_pass:
        records.append(dict(municipality=municipality, date=date,
                            name=name, status=status))


def parse_file(path: Path, municipality: str,
               known_names: set, collect_pass: bool = False) -> list:
    """
    Parse one .md file and return a list of participant dicts.

    Parameters
    ----------
    known_names   Mutable set updated in-place as new names are discovered.
    collect_pass  If True, only populate known_names; don't emit records.
    """
    try:
        text = path.read_text(encoding='utf-8')
    except UnicodeDecodeError:
        text = path.read_text(encoding='latin-1')

    text  = normalise_ocr(text)           # Fix C: OCR artefacts
    date  = extract_date(text)
    lines = text.splitlines()
    records = []

    # ── Classify every pipe-containing line ──────────────────────────────
    pipe_lines = [l for l in lines
                  if '|' in l and not re.match(r'^[\s|:\-]+$', l)]

    # Fix B: label-row table (Indre Østfold) must be detected FIRST so that
    # the 'Medl…' text in col[1] doesn't trigger has_navn_table.
    has_label_row_table = any(
        re.search(r'^\|\s*(til\s+stede|forfall)\s*\|', l, re.I)
        for l in pipe_lines
    )

    # Fix A + Fix N: detect 'Medl…' / 'Navn' column header.
    # Only match short column-header cells (< 25 chars), not full sentences
    # containing "medlemmene" etc. (Alvdal: agenda table body mentioned it)
    def _has_name_column_header(lines):
        for l in lines:
            if '|' not in l:
                continue
            for cell in l.split('|'):
                cell = cell.strip()
                if len(cell) < 25 and re.search(r'\b(navn|medl[ei]mm)', cell, re.I):
                    return True
        return False

    has_navn_table = (
        not has_label_row_table and _has_name_column_header(lines)
    )

    has_grid_table = (
        not has_label_row_table and
        any(re.search(r'til\s*stede|tilstede\s+p\xe5\s+m\xf8tet', l, re.I)
            for l in pipe_lines)
    )

    # ── Label-row table ('PLACEHOLDER') ──────────────────────────────────
    if has_label_row_table:
        for name, status in parse_label_row_table(lines):
            if name in EXCLUDED_NAMES:
                continue
            _emit(records, collect_pass, municipality, date, name, status, known_names)
        return records

    # ── Standard Navn/Funksjon table ('PLACEHOLDER') ────────
    if has_navn_table:
        for name, status in parse_standard_table(lines):
            if name in EXCLUDED_NAMES:
                continue
            _emit(records, collect_pass, municipality, date, name, status, known_names)
        return records

    # ── Attendance grid ('PLACEHOLDER') ─────────────────────────────
    if has_grid_table:
        grid_start = next(
            (i for i, l in enumerate(lines)
             if re.search(r'til\s*stede|tilstede', l, re.I) and '|' in l),
            None,
        )
        if grid_start is not None:
            grid_block = []
            for l in lines[grid_start:]:
                if '|' in l or re.match(r'^[\s|:\-]+$', l) or not l.strip():
                    grid_block.append(l)
                else:
                    break
            for name, status in parse_attendance_grid(grid_block):
                if name in EXCLUDED_NAMES:
                    continue
                _emit(records, collect_pass, municipality, date, name, status, known_names)
        return records

    # ── Text / list parser ────────────────────────────────────────────────
    section     = 'unknown'
    pending_hdr = ''
    names_seen  = 0   # Fix F: don't leave 'member' section before first name

    i = 0  # index-based loop enables lookahead (Fix H)
    while i < len(lines):
        raw_line   = lines[i]
        line       = raw_line.strip()
        clean_line = re.sub(r'^#+\s*', '', line).strip()
        i += 1

        if not clean_line:
            continue

        # ── Complete a pending split header ──────────────────────────────
        if pending_hdr:
            combined = f"{pending_hdr} {clean_line}"
            sec = classify(combined)
            if sec != 'unknown':
                if sec == 'end':
                    break
                # Fix F: guard against premature section change
                if sec in ('absent', 'admin') and section == 'member' and names_seen == 0:
                    pending_hdr = ''
                    continue
                section = sec
                pending_hdr = ''
                # Fix J: fall through — process clean_line as participant data
                # if it contains names ('PLACEHOLDER': names follow the split header
                # line directly, so discarding with 'continue' would lose them)
            else:
                pending_hdr = ''
                continue   # combined unrecognised; skip this continuation line

        # ── Classify current line ─────────────────────────────────────────
        sec = classify(clean_line)
        if sec != 'unknown':
            if sec == 'end':
                break
            # Fix F: don't abandon 'member' before seeing any names
            if sec in ('absent', 'admin') and section == 'member' and names_seen == 0:
                continue
            section = sec
            if _SPLIT_HDR_START.match(clean_line.lower()) and len(clean_line) < 30:
                pending_hdr = clean_line
            continue

        # ── Possible split-header start ───────────────────────────────────
        if _SPLIT_HDR_START.match(clean_line.lower()) and len(clean_line) < 30:
            pending_hdr = clean_line
            continue

        # ── Only process participant sections ─────────────────────────────
        if section not in ('member', 'vara'):
            continue

        default_status = 'varamedlem' if section == 'vara' else 'medlem'

        # Inline (vara) marker
        if re.search(r'\(vara\)', clean_line, re.I):
            default_status = 'varamedlem'
            clean_line = re.sub(r'\(vara\)', '', clean_line, flags=re.I).strip()

        # Fix G: (ABBREV) group delimiter — e.g. 'Name (UFR) Name (UFR)'
        if _PAREN_DELIM.search(clean_line):
            parts = [clean_name(p) for p in _PAREN_DELIM.split(clean_line) if clean_name(p)]
            if len(parts) >= 2 and all(is_name(p) for p in parts):
                for p in parts:
                    if p not in EXCLUDED_NAMES:
                        names_seen += 1
                        _emit(records, collect_pass, municipality, date,
                              p, default_status, known_names)
                continue

        # Fix E: comma-separated names — try simple split first, then Last,First
        if ',' in clean_line and not clean_line.endswith(':'):
            # Stage 1: 'First Last, First Last' — all comma-parts are valid names
            parts = [clean_name(p) for p in clean_line.split(',') if clean_name(p)]
            if len(parts) >= 2 and all(is_name(p) for p in parts):
                for n in parts:
                    if n not in EXCLUDED_NAMES:
                        names_seen += 1
                        _emit(records, collect_pass, municipality, date,
                              n, default_status, known_names)
                continue
            # Stage 2: 'Last, First' ('PLACEHOLDER' style)
            lf_names = split_last_first_line(clean_line)
            if lf_names:
                for n in lf_names:
                    if n not in EXCLUDED_NAMES:
                        names_seen += 1
                        _emit(records, collect_pass, municipality, date,
                              n, default_status, known_names)
                continue

        candidate  = clean_name(clean_line)
        if not candidate or candidate in EXCLUDED_NAMES:
            continue

        word_count = len(candidate.split())

        if is_name(candidate):
            # Fix H + M: lookahead for inline status label on next non-empty line
            # ('PLACEHOLDER': 'Name\nVaramedlem'; B'PLACEHOLDER': 'Name\nMedlem')
            j = i  # i was already incremented
            while j < len(lines) and not lines[j].strip():
                j += 1
            if j < len(lines):
                nxt = re.sub(r'^#+\s*', '', lines[j].strip()).strip().lower()
                nxt_c = re.sub(r'\s+', '', nxt)
                if nxt_c in ('varamedlem', 'varamedlemmar', 'varamedlemmer'):
                    default_status = 'varamedlem'
                elif nxt_c in ('medlem', 'leder', 'nestleder', 'leiar', 'nestleiar'):
                    default_status = 'medlem'

            # 5+ word single name may be two concatenated — try to split
            if word_count >= 5 and known_names:
                parts = split_with_known_names(candidate, known_names)
                if len(parts) > 1 and all(is_name(p) for p in parts):
                    for p in parts:
                        if p not in EXCLUDED_NAMES:
                            names_seen += 1
                            _emit(records, collect_pass, municipality, date,
                                  p, default_status, known_names)
                    continue

            names_seen += 1
            _emit(records, collect_pass, municipality, date,
                  candidate, default_status, known_names)
            continue

        # Too many words — try known_names split
        if word_count > 4 and known_names:
            parts = split_with_known_names(candidate, known_names)
            for p in parts:
                p = clean_name(p)
                if is_name(p) and p not in EXCLUDED_NAMES:
                    names_seen += 1
                    _emit(records, collect_pass, municipality, date,
                          p, default_status, known_names)

    # ── Fix D: Headerless fallback ('PLACEHOLDER') ──────────────────────────────
    # If we parsed nothing and never detected a member section, scan for the
    # first comma-name line before the agenda and treat it as member names.
    if not records and section == 'unknown':
        for raw_line in lines:
            cl = re.sub(r'^#+\s*', '', raw_line.strip()).strip()
            if classify(cl) == 'end':
                break
            if ',' not in cl:
                continue
            parts = [clean_name(p) for p in cl.split(',') if clean_name(p)]
            if len(parts) >= 2 and all(is_name(p) for p in parts):
                for p in parts:
                    if p not in EXCLUDED_NAMES:
                        _emit(records, collect_pass, municipality, date,
                              p, 'medlem', known_names)
                break   # take only the first qualifying line

    return records


In [ ]:
# ── Cell 10 ─ Two-pass processing  (all files per municipality) ──────────────
#
# known_names grows throughout both passes across all files.
# Every new name seen in any file becomes available to split
# concatenated lines in subsequent files.

known_names = set()

# ── Pass 1 ── collect names only (no CSV output) ──────────────────────────
print("Pass 1 – collecting known names...")
for municipality, paths in md_files.items():
    for path in paths:
        parse_file(path, municipality, known_names, collect_pass=True)
print(f"  -> {len(known_names)} unique names collected\n")

# ── Pass 2 ── extract all participants ────────────────────────────────────
print("Pass 2 – extracting participants...")
all_records = []
for municipality, paths in md_files.items():
    recs = []
    for path in paths:
        recs.extend(parse_file(path, municipality, known_names, collect_pass=False))
    all_records.extend(recs)
    print(f"  {municipality:<35}  {len(recs):4d} participants  "
          f"({len(paths)} file(s))")

print(f"\nTotal records: {len(all_records)}")

In [ ]:
# ── Cell 11 ─ Build DataFrame, save CSV ─────────────────────────────────────

df = pd.DataFrame(all_records, columns=['municipality', 'date', 'name', 'status'])
n_raw = len(df)

# ── Belt-and-suspenders exclusion filter ─────────────────────────────────
if EXCLUDED_NAMES:
    df = df[~df['name'].isin(EXCLUDED_NAMES)]
    n_excluded = n_raw - len(df)
    if n_excluded:
        print(f"Removed {n_excluded} rows matching EXCLUDED_NAMES")

# ── Derive year from date (first 4 characters) ────────────────────────────
df['year'] = df['date'].str[:4].where(df['date'].str.len() >= 4, other='????')

# ── Normalise status ──────────────────────────────────────────────────────
df['status'] = df['status'].str.lower().str.strip()
df.loc[~df['status'].isin(['medlem', 'varamedlem']), 'status'] = 'ukjent'

# ── Drop rows with unrecognised status ────────────────────────────────────
n_before_filter = len(df)
df = df[df['status'].isin(['medlem', 'varamedlem'])]
n_dropped = n_before_filter - len(df)
if n_dropped:
    print(f"Dropped {n_dropped} rows with status 'ukjent'")

# ── Remove exact duplicate rows ───────────────────────────────────────────
n_before_dedup = len(df)
df.drop_duplicates(subset=['municipality', 'date', 'name', 'status'], inplace=True)
n_dupes = n_before_dedup - len(df)
if n_dupes:
    print(f"Removed {n_dupes} exact duplicate rows")

# ── Sort ──────────────────────────────────────────────────────────────────
df.sort_values(['municipality', 'year', 'date', 'status', 'name'], inplace=True)
df.reset_index(drop=True, inplace=True)

# ── Flag names with more than 3 words for review ─────────────────────────
df['revise'] = df['name'].apply(lambda n: 'yes' if len(n.split()) > 3 else '')
n_revise = (df['revise'] == 'yes').sum()
if n_revise:
    print(f"Flagged {n_revise} rows with >3-word names for review")

# ── Explicit column order ────────────────────────────────────────────────
df = df[['municipality', 'date', 'year', 'name', 'status', 'revise']]

# ── Save combined CSV ──────────────────────────────────────────────────────
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f"\nSaved combined -> {OUTPUT_CSV}")
print(f"Rows  : {len(df)}  (from {n_raw} raw records)")

# ── Save per-municipality CSVs ────────────────────────────────────────────
print("\nSaving per-municipality CSVs...")
for muni_name in df['municipality'].unique():
    muni_df = df[df['municipality'] == muni_name]
    # Sanitise municipality name for filename (lowercase, replace spaces)
    safe_name = muni_name.lower().replace(' ', '_')
    muni_csv = BASE_DIR / muni_name / f"participants_{safe_name}.csv"
    muni_csv.parent.mkdir(parents=True, exist_ok=True)
    muni_df.to_csv(muni_csv, index=False, encoding='utf-8')
    print(f"  {muni_name:<35}  {len(muni_df):4d} rows -> {muni_csv.name}")

df.head(30)

In [ ]:
# ── Cell 12 ─ Quality check ──────────────────────────────────────────────────

summary = (
    df.groupby('municipality')
    .agg(
        total         = ('name',   'count'),
        members       = ('status', lambda s: (s == 'medlem').sum()),
        varamedlemmer = ('status', lambda s: (s == 'varamedlem').sum()),
        meetings      = ('date',   'nunique'),
    )
    .sort_values('total')
)
print(summary.to_string())

missing = set(md_files) - set(df['municipality'].unique())
if missing:
    print("\nWARNING – no participants found for:", sorted(missing))
else:
    print("\nAll municipalities produced at least one participant.")

In [ ]:
# ── Cell 13 ─ Inspect one municipality ──────────────────────────────────────

inspect = "'PLACEHOLDER'"   # <── change to any folder name

sub = df[df['municipality'] == inspect].sort_values(['date', 'status', 'name'])
print(f"{inspect}: {len(sub)} rows across {sub['date'].nunique()} meeting(s)")
print(sub.to_string(index=False))

In [ ]:
# ── Cell 14 ─ Debug helper: show raw first lines of any file ─────────────────
#
# Useful for investigating why a file is not parsed correctly.

debug_municipality = "'PLACEHOLDER'"   # <── municipality folder name
debug_file_index   = 0         # <── 0 = first file, 1 = second, etc.

paths = md_files.get(debug_municipality)
if not paths:
    print(f"'{debug_municipality}' not found.")
elif debug_file_index >= len(paths):
    print(f"Only {len(paths)} file(s) in '{debug_municipality}'.")
else:
    path = paths[debug_file_index]
    print(f"File: {path.name}\n")
    for i, l in enumerate(path.read_text(encoding='utf-8').splitlines()[:80], 1):
        print(f"{i:3d}  {l}")